In [2]:
from pathlib import Path

import pandas as pd

from processing import get_audio_and_speech_data as gaasd
from processing import get_segment_information as gsi


OUTPUT_DIR = Path("cleaned_data")
OUTPUT_BASENAME_CANDIDATE = "candidate_segments_with_speech"
OUTPUT_BASENAME_HOST = "host_segments_with_speech"
RELATIVE_AMBIGUITY_THRESHOLD = 0.2


This section creates a dataframe that names each segment with the belonging candidate for that segment and removes some segments based on a AMBIGUITY THRESHOLD. that is if its not too centered its most probably two pepole talking over each other


In [3]:
def build_candidate_segments_with_speech() -> pd.DataFrame:
    audio_file_names = gaasd.get_audio_file_names()
    candidates = gaasd.get_candidate_names(audio_file_names)

    all_candidate_clusters = pd.concat(
        [
            gsi.infer_candidate_clusters(
                candidate_name,
                audio_file_names,
                RELATIVE_AMBIGUITY_THRESHOLD,
            )
            for candidate_name in candidates
        ],
        ignore_index=True,
    )

    all_candidate_segments_dataframe = gsi.get_candidate_segments(all_candidate_clusters, RELATIVE_AMBIGUITY_THRESHOLD)

    return gsi.add_speech_data_to_candidate_segments(all_candidate_segments_dataframe)


def save_dataframe(dataframe: pd.DataFrame, OUTPUT_BASENAME) -> None:
    OUTPUT_DIR.mkdir(exist_ok=True)

    pickle_path = OUTPUT_DIR / f"{OUTPUT_BASENAME}.pkl"
    csv_path = OUTPUT_DIR / f"{OUTPUT_BASENAME}.csv"

    dataframe.to_pickle(pickle_path)
    dataframe.to_csv(csv_path, index=False)

    print(f"Saved pickle: {pickle_path}")
    print(f"Saved CSV: {csv_path}")


In [4]:
candidate_segments_with_speech = build_candidate_segments_with_speech()
save_dataframe(candidate_segments_with_speech, OUTPUT_BASENAME_CANDIDATE)



Saved pickle: cleaned_data/candidate_segments_with_speech.pkl
Saved CSV: cleaned_data/candidate_segments_with_speech.csv


In [5]:
print(len(candidate_segments_with_speech))

1982


In [6]:
import importlib
import processing.get_segment_information as gsi

gsi = importlib.reload(gsi)

In [7]:
audio_file_names = gaasd.get_audio_file_names()

host_segments_dataframe = gsi.get_host_segments_dataframe(audio_file_names, RELATIVE_AMBIGUITY_THRESHOLD)

host_text_dataframe = host_segments_dataframe.loc[
    host_segments_dataframe["has_transcript"],
    ["debate_name", "audio_file_name", "time stamp", "duration", "transcript"]
]


print(len(host_segments_dataframe))
#host_segments_dataframe
host_text_dataframe

save_dataframe(host_segments_dataframe, OUTPUT_BASENAME_HOST)


490
Saved pickle: cleaned_data/host_segments_with_speech.pkl
Saved CSV: cleaned_data/host_segments_with_speech.csv


In [9]:
audio_file_names = gaasd.get_audio_file_names()
candidates = gaasd.get_candidate_names(audio_file_names)

all_candidate_clusters = pd.concat(
    [
        gsi.infer_candidate_clusters(
            candidate_name,
            audio_file_names,
            RELATIVE_AMBIGUITY_THRESHOLD,
        )
        for candidate_name in candidates
    ],
    ignore_index=True,
)

cluster_scores = all_candidate_clusters[
    [
        "candidate_name",
        "debate_name",
        "speaker_cluster",
        "candidate_similarity_score",
        "second_speaker_cluster",
        "second_similarity_score",
        "similarity_score_margin",
    ]
].copy()

cluster_scores[
    ["candidate_similarity_score", "second_similarity_score", "similarity_score_margin"]
] = cluster_scores[
    ["candidate_similarity_score", "second_similarity_score", "similarity_score_margin"]
].round(3)

display(cluster_scores.sort_values(["candidate_name", "debate_name"]))

,candidate_name,debate_name,speaker_cluster,candidate_similarity_score,second_speaker_cluster,second_similarity_score,similarity_score_margin
35,Cotrim_Figueiredo,Cotrim_Figueiredo_vs_Filipe_November_30,2,0.924,1,0.322,0.602
36,Cotrim_Figueiredo,Cotrim_Figueiredo_vs_Gouveia_Melo_November_20,2,0.892,1,0.351,0.540
37,Cotrim_Figueiredo,Cotrim_Figueiredo_vs_Marques_Mendes_December_7,3,0.871,1,0.349,0.522
38,Cotrim_Figueiredo,Cotrim_Figueiredo_vs_Ventura_December_19,2,0.900,1,0.433,0.467
39,Cotrim_Figueiredo,Martins_vs_Cotrim_Figueiredo_December_4,1,0.917,2,0.219,0.698
40,Cotrim_Figueiredo,Pinto_vs_Cotrim_Figueiredo_November_24,2,0.888,1,0.305,0.583
41,Cotrim_Figueiredo,Seguro_vs_Cotrim_Figueiredo_December_17,2,0.894,1,0.522,0.372
0,Filipe,Cotrim_Figueiredo_vs_Filipe_November_30,1,0.924,2,0.323,0.601
1,Filipe,Filipe_vs_Gouveia_Melo_December_2,2,0.929,1,0.346,0.583
2,Filipe,Filipe_vs_Marques_Mendes_November_18,1,0.925,2,0.283,0.642
